# KnowSeek.Ai — Control Hub
**Version: rev04_work — Branch: main_sia_04**

> This notebook is the **window**. The `.py` files are the **engine**.
> All logic lives in `01_backend/modules/`. This notebook calls those files and shows the results.

---

## Chapter Overview

| Chapter | Topic | Status |
|---------|-------|--------|
| 1 | Environment Check | |
| 2 | Load Data | |
| 3 | EDA — Explore the Data | |
| 4 | BM25 Baseline | |
| 5 | RAG + llama3 | |
| 6 | Compare BM25 vs RAG | |

---

**How to use this notebook:**
- Run chapters from top to bottom
- Each chapter can be run independently
- Add `🔴` to a section title if it is not finished in time
- Add `✅` to a section title when it is done

---
# Chapter 1 — Environment Check
> **Goal:** Make sure all tools are running before we start.

We check three things:
- Python environment is active
- Ollama is running with the right models
- ChromaDB folder is ready

## 1.1 Virtual Environment Check
**Task:** Check if Python is the right version and all packages are installed.

In [ ]:
import sys
import importlib

print(f"Python version: {sys.version}")
print(f"Expected:       3.11.3")
print()

packages = [
    "langchain", "chromadb", "rank_bm25",
    "mlflow", "pdfplumber", "fastapi",
    "pandas", "seaborn", "plotly"
]

for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"  OK  {pkg}")
    except ImportError:
        print(f"  MISSING  {pkg}")

## 1.2 Ollama Connection Check
**Task:** Check if Ollama is running and the models are available.

In [ ]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("Ollama is running")
    print()
    for m in models:
        print(f"  {m}")
    print()
    for required in ["llama3", "nomic-embed-text"]:
        found = any(required in m for m in models)
        print(f"  {'OK' if found else 'MISSING'}  {required}")
except Exception as e:
    print(f"Ollama not running — start with: brew services start ollama")
    print(f"Error: {e}")

## 1.3 ChromaDB Setup Check
**Task:** Make sure the ChromaDB folder exists and is ready to store data.

In [ ]:
import chromadb
from pathlib import Path

DB_PATH = "./chroma_db"
Path(DB_PATH).mkdir(exist_ok=True)

client = chromadb.PersistentClient(path=DB_PATH)
collections = client.list_collections()

print(f"ChromaDB path:   {DB_PATH}")
print(f"Collections:     {len(collections)}")
for c in collections:
    print(f"  - {c.name}")
if not collections:
    print("  (empty — will be filled in Chapter 2)")

---
# Chapter 2 — Load Data
> **Goal:** Load all PDF files from `05_data/` and prepare them for the AI.

We do four steps:
1. Find all PDF files
2. Split the text into small chunks
3. Add metadata to every chunk
4. Save everything in ChromaDB

## 2.1 File Loader
**Task:** Find all PDF files in `05_data/` and show what we have.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../05_data")

files = []
for f in DATA_PATH.rglob("*"):
    if f.suffix.lower() in [".pdf", ".png", ".webp", ".jpg"]:
        files.append({
            "filename": f.name,
            "folder": f.parent.name,
            "type": f.suffix.lower(),
            "size_kb": round(f.stat().st_size / 1024, 1)
        })

df_files = pd.DataFrame(files)
print(f"Total files found: {len(df_files)}")
print()
print(df_files.to_string(index=False))

## 2.2 Text Splitting (Chunking)
**Task:** Cut the PDF text into small pieces so the AI can read them easily.

This calls `ingest.py` from the backend.

In [ ]:
import sys
sys.path.append("../../01_backend/modules/02_docseek")

# import ingest
# chunks = ingest.load_and_chunk(DATA_PATH)
# print(f"Total chunks created: {len(chunks)}")

print("ingest.py not ready yet — coming in next step")
print("This cell will call: ingest.load_and_chunk()")

## 2.3 Tagging & Metadata
**Task:** Add an ID, date, OEM code, and category to every chunk.

Every chunk gets metadata like this:
```python
metadata = {
    'source_id': '#74',
    'filename':  'OEM-V_SPEC_001.pdf',
    'page':      14,
    'oem_code':  'OEM-V',
    'category':  'Corrosion',
    'language':  'DE'
}
```

In [ ]:
print("Metadata tagging will be added here")
print("This cell will call: ingest.add_metadata()")

## 2.4 Vectorization & Storage
**Task:** Convert text chunks into vectors and save them in ChromaDB.

This calls `embed.py` from the backend.

In [ ]:
print("Vectorization will be added here")
print("This cell will call: embed.vectorize_and_store()")

---
# Chapter 3 — EDA — Explore the Data
> **Goal:** Understand the data before we start building the AI system.

We look at:
- How many files per folder?
- What file types do we have?
- How long are the documents?
- What languages are used?

## 3.0 First Ingest Results — 14.03.2026 main_sia05

| Metric | Value |
|--------|-------|
| PDFs | 7 |
| Total pages | 13 |
| Total chunks | 47 |
| Chunk size | 500 / overlap 100 |
| Ingest time | 3.67s |
| Embedding | nomic-embed-text via Ollama |
| Collection | docseek |

**Key findings:**
- All documents are 1-2 pages — doc_type = Datasheet
- No Lastenheft in test data yet
- 1 German document (OEM Vergleich) — 6 English
- Chunk config "medium" works well for this data size

## 3.1 Data Overview
**Task:** Show how many files we have per category and file type.

In [ ]:
import plotly.express as px
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../05_data")

files = []
for f in DATA_PATH.rglob("*"):
    if f.suffix.lower() in [".pdf", ".png", ".webp", ".jpg"]:
        files.append({
            "folder": f.parent.name,
            "type": f.suffix.lower()
        })

df = pd.DataFrame(files)

fig = px.histogram(
    df, x="folder", color="type",
    title="Files per folder",
    labels={"folder": "Folder", "count": "Number of files"},
    barmode="group"
)
fig.show()

## 3.2 PDF Analysis
**Task:** Show how many pages each PDF has and how big it is.

In [ ]:
import pdfplumber
import pandas as pd
import plotly.express as px
from pathlib import Path

DATA_PATH = Path("../../05_data")

pdf_data = []
for f in DATA_PATH.rglob("*.pdf"):
    try:
        with pdfplumber.open(f) as pdf:
            pdf_data.append({
                "filename": f.name,
                "folder": f.parent.name,
                "pages": len(pdf.pages),
                "size_kb": round(f.stat().st_size / 1024, 1)
            })
    except Exception as e:
        print(f"Could not read {f.name}: {e}")

df_pdf = pd.DataFrame(pdf_data)
print(df_pdf.to_string(index=False))
print()
print(f"Total PDFs:  {len(df_pdf)}")
print(f"Total pages: {df_pdf['pages'].sum()}")

fig = px.bar(
    df_pdf, x="filename", y="pages",
    color="folder",
    title="Pages per PDF"
)
fig.update_xaxes(tickangle=30)
fig.show()

## 3.3 Chunk Size Analysis
**Task:** Show how the text is split into chunks — how big are the pieces?

In [ ]:
print("Chunk analysis will be added here")
print("This cell will run after ingest.py is ready")
print("It will show: chunk size distribution per document type")

## 3.4 Language Distribution
**Task:** Show how many documents are in German vs English.

In [ ]:
import plotly.express as px
import pandas as pd

lang_data = [
    {"file": "GEORGE-01", "language": "EN"},
    {"file": "MICKEY-01", "language": "EN"},
    {"file": "MICKEY-01 rev02", "language": "EN"},
    {"file": "ZEUS-01", "language": "EN"},
    {"file": "OEM Vergleich", "language": "DE"},
    {"file": "HADES-59", "language": "EN"},
    {"file": "SWIFT-01", "language": "EN"},
]

df_lang = pd.DataFrame(lang_data)
fig = px.pie(
    df_lang, names="language",
    title="Language distribution",
    color_discrete_map={"DE": "#10B981", "EN": "#0EA5E9"}
)
fig.show()

---
# Chapter 4 — BM25 Baseline
> **Goal:** Test the old keyword search (BM25) so we can compare it to the AI search.

BM25 is the baseline model — it searches by keywords, not by meaning.
We measure: how good are the results? How fast is it?

## 4.1 BM25 Setup
**Task:** Load all text and build the BM25 index.

In [ ]:
from rank_bm25 import BM25Okapi
import time

corpus = [
    "Salt spray test requirements for automotive parts",
    "Corrosion performance standard for exterior components",
    "Hexagon screw M16 strength class 8.8 zinc coating",
    "Paint adhesion test cathodic e-coating specification",
    "Interior structure corrosion requirements OEM standard"
]

tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)

print(f"BM25 index built")
print(f"Documents indexed: {len(corpus)}")

## 4.2 BM25 Search Test
**Task:** Run test queries and measure the results + time.

In [ ]:
import time
import pandas as pd

test_queries = [
    "salt spray test OEM",
    "screw M16 corrosion",
    "paint coating standard"
]

results = []
for query in test_queries:
    start = time.time()
    scores = bm25.get_scores(query.lower().split())
    best_idx = scores.argmax()
    elapsed = round((time.time() - start) * 1000, 2)

    results.append({
        "query": query,
        "best_match": corpus[best_idx][:50],
        "score": round(float(scores[best_idx]), 3),
        "time_ms": elapsed
    })

df_bm25 = pd.DataFrame(results)
print(df_bm25.to_string(index=False))

## 4.3 Log BM25 Results to MLFlow
**Task:** Save the BM25 results in MLFlow so we can compare later.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("KnowSeek — BM25 vs RAG")

with mlflow.start_run(run_name="BM25 Baseline"):
    mlflow.log_param("model", "BM25")
    mlflow.log_param("documents", len(corpus))
    mlflow.log_metric("avg_score", df_bm25["score"].mean())
    mlflow.log_metric("avg_time_ms", df_bm25["time_ms"].mean())

print("BM25 results logged to MLFlow")
print(f"Average score:   {round(df_bm25['score'].mean(), 3)}")
print(f"Average time ms: {round(df_bm25['time_ms'].mean(), 2)}")

---
# Chapter 5 — RAG + llama3
> **Goal:** Test the AI search (RAG) and measure how good it is.

RAG = Retrieval Augmented Generation
- Step 1: Find the right chunks in ChromaDB
- Step 2: Send them to llama3
- Step 3: Get a clean answer with source

## 5.1 Prompt Engineering
**Task:** Build the system prompt that tells llama3 how to behave.

In [ ]:
SYSTEM_PROMPT = """
You are a professional engineering assistant.
You only answer based on the documents provided.
Always include the source document and page number.
If you are not sure, say so — do not guess.
Keep your answer short and clear.
"""

print("System prompt ready")
print(SYSTEM_PROMPT)

## 5.2 RAG Search + Answer Generation
**Task:** Run a test query through the full RAG pipeline and measure the time.

This calls `search.py` and `answer.py` from the backend.

In [ ]:
print("RAG pipeline will be added here")
print("This cell will call:")
print("  search.find_chunks(query)")
print("  answer.generate(chunks, query)")
print()
print("Returns: answer + source + confidence score")

## 5.3 Source Attribution
**Task:** Make sure every answer shows exactly where the information comes from.

In [ ]:
print("Source attribution will be added here")
print("Every answer will show: filename + page + confidence score")

## 5.4 Log RAG Results to MLFlow
**Task:** Save the RAG results in MLFlow so we can compare with BM25.

In [ ]:
print("MLFlow RAG logging will be added here")
print("This cell will log: model=RAG, avg_score, avg_time_ms")

---
# Chapter 6 — Compare BM25 vs RAG
> **Goal:** Show clearly that RAG is better than BM25.
> This is the key chart for the Midterm PPT.

We compare:
- Answer quality (score)
- Answer time (ms)
- Confidence distribution

## 6.1 Score Comparison Chart
**Task:** Bar chart — BM25 score vs RAG score per query.

In [ ]:
import plotly.express as px
import pandas as pd

comparison = pd.DataFrame([
    {"query": "salt spray test",   "model": "BM25", "score": 0.42},
    {"query": "salt spray test",   "model": "RAG",  "score": 0.89},
    {"query": "screw M16",         "model": "BM25", "score": 0.38},
    {"query": "screw M16",         "model": "RAG",  "score": 0.91},
    {"query": "paint coating",     "model": "BM25", "score": 0.45},
    {"query": "paint coating",     "model": "RAG",  "score": 0.87},
])

fig = px.bar(
    comparison, x="query", y="score",
    color="model", barmode="group",
    title="BM25 vs RAG — Answer Quality",
    color_discrete_map={"BM25": "#888780", "RAG": "#10B981"}
)
fig.show()

print("Note: replace with real MLFlow results when available")

## 6.2 Answer Time Chart
**Task:** Bar chart — how fast is BM25 vs RAG?

In [ ]:
import plotly.express as px
import pandas as pd

timing = pd.DataFrame([
    {"model": "BM25", "avg_time_ms": 12},
    {"model": "RAG + llama3", "avg_time_ms": 2400},
])

fig = px.bar(
    timing, x="model", y="avg_time_ms",
    title="Answer Time — BM25 vs RAG (ms)",
    color="model",
    color_discrete_map={"BM25": "#888780", "RAG + llama3": "#10B981"}
)
fig.show()

print("Note: replace with real MLFlow results when available")

## 6.3 Confidence Score Distribution
**Task:** Show how confident the AI is across all test queries.

In [ ]:
import plotly.express as px
import pandas as pd

confidence = pd.DataFrame([
    {"query": "salt spray test OEM-V",     "confidence": 0.92, "signal": "Green"},
    {"query": "corrosion standard MICKEY",  "confidence": 0.87, "signal": "Green"},
    {"query": "screw M16 torque",           "confidence": 0.71, "signal": "Yellow"},
    {"query": "paint adhesion interior",    "confidence": 0.65, "signal": "Yellow"},
    {"query": "unknown part XYZ",           "confidence": 0.41, "signal": "Red"},
])

fig = px.bar(
    confidence, x="query", y="confidence",
    color="signal",
    title="Confidence Score per Query",
    color_discrete_map={"Green": "#10B981", "Yellow": "#F59E0B", "Red": "#EF4444"}
)
fig.update_xaxes(tickangle=20)
fig.add_hline(y=0.85, line_dash="dash", line_color="#10B981", annotation_text="Green threshold 85%")
fig.add_hline(y=0.60, line_dash="dash", line_color="#F59E0B", annotation_text="Yellow threshold 60%")
fig.show()

---
*KnowSeek.Ai — Version rev04_work — Branch main_sia_04*

*All data stays local. No cloud. No internet required.*